# SDS — OCR

Run OCR on generated documents and evaluate quality.

**Requires:** dataset generated by `generate.ipynb`

## 1. Configuration

In [ ]:
import sys
from pathlib import Path

PROJECT = Path(".").resolve()
if not (PROJECT / "synthdoc").exists():
    PROJECT = PROJECT.parent
DATA_DIR = PROJECT / "data"

sys.path.insert(0, str(PROJECT))

In [ ]:
OCR_ENGINE = "glm"  # glm | tesseract | qwenvl | deepseek

## 2. Load Pages

In [ ]:
from synthdoc.bench.helpers import load_ner_pages, FULL_ENTITY_TYPES

pages = load_ner_pages(DATA_DIR, entity_types=FULL_ENTITY_TYPES)

print(f"Loaded {len(pages)} pages")
print(f"  Documents: {len(set(p['doc_id'] for p in pages))}")
print(f"  Templates: {len(set(p['template_type'] for p in pages))}")

## 3. Run OCR

In [ ]:
from synthdoc.bench.config import load_engines, _resolve, _dispatch_ocr
from synthdoc.bench.run_benchmark import _load_ocr_cache, _save_ocr_cache
from synthdoc.bench.helpers import InferenceTimer

engines_config = load_engines()
runner, model_id = _resolve(OCR_ENGINE, "ocr", engines_config)
ocr_key = f"{OCR_ENGINE}_ocr"

_load_ocr_cache(pages, ocr_key, DATA_DIR)
remaining = [p for p in pages if ocr_key not in p]

if remaining:
    print(f"Running {OCR_ENGINE} OCR on {len(remaining)}/{len(pages)} pages")
    with InferenceTimer(f"{OCR_ENGINE}_ocr") as timer:
        _dispatch_ocr(runner, remaining, model_id=model_id)
    _save_ocr_cache(pages, ocr_key, DATA_DIR)
    print(f"Done in {timer.result['time_s']}s")
else:
    print(f"All {len(pages)} pages already cached")

## 4. Quality Report

In [ ]:
import pandas as pd
from synthdoc.bench.helpers import compute_wer, compute_cer

rows = []
for p in pages:
    ocr_text = p.get(ocr_key, "")
    gt_text = p.get("gt_text", "")
    rows.append({
        "stem": p["stem"],
        "template": p["template_type"],
        "wer": compute_wer(gt_text, ocr_text),
        "cer": compute_cer(gt_text, ocr_text),
    })

ocr_df = pd.DataFrame(rows)
print(f"OCR Quality ({OCR_ENGINE}):")
print(f"  Mean WER: {ocr_df['wer'].mean():.4f}")
print(f"  Mean CER: {ocr_df['cer'].mean():.4f}")
print()
print(ocr_df.groupby("template")[["wer", "cer"]].mean().round(4).to_string())